In [13]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from mne import read_labels_from_annot
from mne.viz import Brain
from mne.datasets import fetch_fsaverage
from nilearn.plotting import plot_surf_stat_map
import os
from pathlib import Path
from matplotlib.colors import Normalize

# Step 1: Load the simplified z-scores matrix from the .mat file
file_path = '/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/mi/mi_time_zscores_matrix.mat'
data = scipy.io.loadmat(file_path)
z_matrix = data['z_matrix']  # z_matrix shape: (68 regions, 4 conditions)

# Step 2: Define the conditions
conditions = ['N1', 'N2', 'N3', 'R_']

# Step 3: Load the Desikan-Killiany atlas from MNE
fsaverage = fetch_fsaverage(verbose=True)
subjects_dir = '/Users/borjan/mne_data/MNE-fsaverage-data'
mne_labels = read_labels_from_annot('fsaverage', parc='aparc', subjects_dir=subjects_dir)



0 files missing from root.txt in /Users/borjan/mne_data/MNE-fsaverage-data
0 files missing from bem.txt in /Users/borjan/mne_data/MNE-fsaverage-data/fsaverage
Reading labels from parcellation...
   read 35 labels from /Users/borjan/mne_data/MNE-fsaverage-data/fsaverage/label/lh.aparc.annot
   read 34 labels from /Users/borjan/mne_data/MNE-fsaverage-data/fsaverage/label/rh.aparc.annot


0.7241379310344828

In [14]:


# Step 5: Initialize the brain visualization
brain = Brain('fsaverage', 'both', surf='pial', subjects_dir=subjects_dir,
              cortex='low_contrast', background='white', size=(800, 600))

# Step 6: Function to add colored labels based on z-scores
def add_colored_labels(brain, labels, scores, cmap, norm):
    for label, score in zip(labels, scores):
        color = cmap(norm(score))[:3]  # Extract RGB values
        brain.add_label(label, color=color, alpha=0.95, borders=False)

# Step 7: Function to generate and save visualizations for each condition
output_dir = Path('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/figures/time/')
output_dir.mkdir(parents=True, exist_ok=True)

def generate_visualization_for_condition(condition_idx, condition_name, z_matrix, brain, output_dir):
    brain.remove_labels()  # Clear previous labels
    
    # Extract z-scores for the current condition (column from the matrix)
    z_scores_for_condition = z_matrix[:, condition_idx]
    
   # Calculate the max absolute value for symmetric color mapping
    abs_max = max(abs(np.min(z_matrix)), np.max(z_matrix))
    vmin = -abs_max  # Symmetric min around zero
    vmax = abs_max   # Symmetric max around zero
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap('coolwarm')

    # Add the z-scores to the brain visualization
    add_colored_labels(brain, mne_labels, z_scores_for_condition, cmap, norm)
    
    # Show the brain from the top view
    brain.show_view('dorsal')
    
    # Save the figure with a unique name
    output_file_path = output_dir / f'mi_time_{condition_name}_brain_viz.png'
    brain.save_image(str(output_file_path))

# Step 7: Loop through conditions and generate visualizations
for i, condition in enumerate(conditions):
    generate_visualization_for_condition(i, condition, z_matrix, brain, output_dir)

# Close the brain visualization once done
brain.close()

# Show the plots
plt.show()